# Joins, subqueries, and CTEs

Normalised data spreads facts across tables, so reading it back means putting the pieces together.
A **join** matches rows between tables on a shared key; a **subquery** nests one query inside
another; a **CTE** (common table expression) names an intermediate result so a query reads like a
recipe. This notebook combines airports, airlines, routes, cities, and countries.

## Learning objectives

By the end of this notebook you will be able to:

- write an `INNER JOIN` and explain what it drops;
- use `LEFT JOIN` to keep rows that have no match;
- join a table to itself for pairs or hierarchies;
- nest a query as a subquery in `WHERE`, `FROM`, or `SELECT`;
- factor a query into a `WITH` clause (CTE) for readability.

## Concept

A join needs a **join condition**, usually `a.key = b.key`. An `INNER JOIN` returns only the
matching pairs; a `LEFT JOIN` returns every row from the left table and fills missing right-hand
columns with `NULL`. That asymmetry is how you find "airports with no routes": keep the airports
and filter where the right side is null.

A **subquery** is a complete `SELECT` used as a value, a table, or a predicate. A correlated
subquery references the outer row and is re-evaluated for each one, which is expressive but can be
slow. A **CTE** (`WITH name AS (...)`) is a named subquery that the rest of the statement can
reuse; it makes multi-step logic readable and is often the clearest option.

Watch for join keys that are not unique: joining on a non-unique column multiplies rows. Counting
before and after a join catches that mistake quickly.

## Worked example

### Connect and preview the join

Each `routes` row stores airport ids, so we join twice to `airports` — once for the origin, once
for the destination — and once to `airlines`.

In [1]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))
from load import build_database, DEFAULT_DB
from ds_practice import connect_sqlite, query
from ds_practice.paths import data_path

db_path = data_path(DEFAULT_DB)
if not db_path.exists():
    db_path = build_database()
conn = connect_sqlite(db_path)

sample = query(conn, """
    SELECT al.name AS airline, src.name AS origin, dst.name AS destination, r.stops
    FROM routes r
    JOIN airlines al ON al.airline_id = r.airline_id
    JOIN airports src ON src.airport_id = r.source_airport_id
    JOIN airports dst ON dst.airport_id = r.destination_airport_id
    LIMIT 5
""")
display(sample)

,airline,origin,destination,stops


### Inner versus left join

An `INNER JOIN` drops airlines that have no routes; a `LEFT JOIN` keeps every airline and shows a
`NULL` route count for the quiet ones. Comparing the two counts makes the difference concrete.

In [2]:
inner = query(conn, """
    SELECT COUNT(*) AS rows_returned
    FROM airlines al JOIN routes r ON r.airline_id = al.airline_id
""")
left = query(conn, """
    SELECT COUNT(*) AS rows_returned
    FROM airlines al LEFT JOIN routes r ON r.airline_id = al.airline_id
""")
print("inner join rows:", int(inner.iloc[0, 0]))
print("left  join rows:", int(left.iloc[0, 0]))

silent = query(conn, """
    SELECT al.name
    FROM airlines al
    LEFT JOIN routes r ON r.airline_id = al.airline_id
    WHERE r.route_id IS NULL
    LIMIT 5
""")
print("airlines with no routes:")
display(silent)

inner join rows: 0
left  join rows: 0
airlines with no routes:


,name


### A self join

The same table on both sides of a join is a **self join**. Here we pair each route's two airports
to list short domestic hops within one country using the normalised city/country tables.

In [3]:
domestic = query(conn, """
    SELECT co.name AS country, COUNT(*) AS routes
    FROM routes r
    JOIN airports src ON src.airport_id = r.source_airport_id
    JOIN airports dst ON dst.airport_id = r.destination_airport_id
    JOIN cities cs  ON cs.city_id = src.city_id
    JOIN cities cd  ON cd.city_id = dst.city_id
    JOIN countries co ON co.country_id = cs.country_id
    WHERE cs.country_id = cd.country_id
    GROUP BY co.country_id
    ORDER BY routes DESC
    LIMIT 8
""")
display(domestic)

,country,routes


### A subquery

A subquery in `WHERE` can compare each row to an aggregate over a filtered set. This finds airports
that handle more routes than the average airport.

In [4]:
above_average = query(conn, """
    SELECT a.name, COUNT(*) AS departures
    FROM routes r
    JOIN airports a ON a.airport_id = r.source_airport_id
    GROUP BY a.airport_id
    HAVING COUNT(*) > (
        SELECT AVG(cnt) FROM (
            SELECT COUNT(*) AS cnt FROM routes GROUP BY source_airport_id
        )
    )
    ORDER BY departures DESC
    LIMIT 5
""")
display(above_average)

,name,departures


### The same logic as a CTE

Naming the inner count in a `WITH` clause makes the threshold explicit and reusable.

In [5]:
cte = query(conn, """
    WITH departures AS (
        SELECT source_airport_id AS airport_id, COUNT(*) AS n
        FROM routes
        GROUP BY source_airport_id
    )
    SELECT a.name, d.n AS departures
    FROM departures d
    JOIN airports a ON a.airport_id = d.airport_id
    WHERE d.n > (SELECT AVG(n) FROM departures)
    ORDER BY departures DESC
    LIMIT 5
""")
display(cte)

,name,departures


## Exercises

1. **Busiest countries.** Join `routes` to the origin airport's city and country and return the
   top ten countries by number of departing routes.
2. **No-service airports.** Use a `LEFT JOIN` to list five airports that appear in `airports` but
   never as a route origin.
3. **CTE refactor.** Rewrite one of the subquery examples above as a `WITH` clause and explain in
   two sentences why the CTE version is easier to read.

## Limitations

Joins are the most expensive operation in most queries, and a missing index on the join column
forces a full scan. A wrong join key can silently duplicate or drop rows, so check the row count
after every join. `LEFT JOIN ... WHERE right.col IS NULL` is the standard "find the missing"
pattern but breaks if the filter is written in the `ON` clause instead. CTEs improve readability
but do not always improve performance, and SQLite may materialise them; check with `EXPLAIN QUERY
PLAN` when a query matters.